# Paper 3 — does the average-subject checkpoint predict any brain?

> **SUPERSEDED. Do not run cells 4 and 5; their result was withdrawn.**
>
> This notebook produced `noise_ceiling.json` with `0.2247`, CI
> `[0.2147, 0.2363]`, labelled "482 parcels". Both defects are in cell 4.
> `STIMULUS` was left at `None`, so the fallback `list(f.keys())[0]` selected
> episode `s01e02a`, not the `s01e01a` that pass 2 predicts, and subjects do
> not even agree on which session an episode sits under. The guard
> `if data.shape[0] > data.shape[1]` cannot fire on a `(482, 1000)` array, so
> the recordings stayed `(timepoints, parcels)` and every correlation ran
> across the parcel axis. Schaefer-1000 has 1000 parcels; the 482 in that
> artifact was the timepoint count.
>
> The measurement now lives in `scripts/paper3_noise_ceiling.py`, which takes
> the episode as an argument, matches the key by name, and decides orientation
> against the atlas size. Corrected: `0.1517`, CI `[0.1484, 0.1551]`, four
> subjects, 1000 of 1000 parcels, 592 timepoints, on `s01e01a`.
>
> Cells 1 and 2 are still useful for inspecting a mirror.

Pass 1 is CPU only and answers two questions before any GPU time is spent:
what is inside the Algonauts 2025 `.h5` files, and how much shared signal is
there to predict.

The second is the noise ceiling, and it is a real result on its own. It is
computed from the recordings alone, with no encoder involved, so it stands
whatever the checkpoint turns out to do. An encoder cannot be expected to beat
the agreement between real brains watching the same thing.

**Attach `ckadirt/algonauts2025nsl` before running.** No accelerator needed.

In [ ]:
import glob
import os

# The dataset is a datalad clone, so the readable copies sit under fmri/ while
# .git/annex holds the same bytes under hashed names. Look at fmri/ only.
roots = sorted(glob.glob('/kaggle/input/**/algonauts_2025.competitors', recursive=True))
print('dataset roots:', roots)
root = roots[0]

subjects = sorted(glob.glob(os.path.join(root, 'fmri', 'sub-*')))
print('subjects:', [os.path.basename(s) for s in subjects])

h5 = sorted(glob.glob(os.path.join(root, 'fmri', '**', '*.h5'), recursive=True))
print(f'\nh5 files: {len(h5)}')
for path in h5:
    size = os.path.getsize(path) / 1e6
    print(f'  {size:8.1f} MB  {os.path.relpath(path, root)}')

In [ ]:
import h5py

# Structure before analysis. A wrong assumption about axis order silently
# transposes every correlation and still produces plausible numbers.
target = [p for p in h5 if 'friends' in p][0]
print('inspecting:', os.path.basename(target))

with h5py.File(target, 'r') as f:
    keys = list(f.keys())
    print('top-level keys:', len(keys))
    for key in keys[:10]:
        item = f[key]
        kind = 'group' if isinstance(item, h5py.Group) else 'dataset'
        detail = list(item.keys())[:6] if kind == 'group' else f'{item.shape} {item.dtype}'
        print(f'  {key:32s} {kind:8s} {detail}')

    first = f[keys[0]]
    if isinstance(first, h5py.Group):
        for sub in list(first.keys())[:6]:
            print(f'    {keys[0]}/{sub}: {first[sub].shape} {first[sub].dtype}')

In [ ]:
%%bash
set -e
mkdir -p /kaggle/temp
cd /kaggle/temp
rm -rf monarch
git clone -q --branch thesis/amendment-and-analysis-layer \
  https://github.com/brn-mwai/monarch.git monarch
cd monarch && git log --oneline -1
# encoder_validation.py carries the noise ceiling and the bootstrap; it is tested
# and must not be reimplemented in a notebook cell.
test -f services/inference/app/services/encoder_validation.py && echo 'validation module present'

In [ ]:
import sys

import numpy as np

sys.path.insert(0, '/kaggle/temp/monarch/services/inference')
from app.services.encoder_validation import bootstrap_ci, noise_ceiling

# One stimulus common to every subject, so the ceiling is computed on responses
# to the same thing. Which key that is comes from the inspection cell above.
STIMULUS = None  # set from the printed keys, e.g. 's01e01a'

responses = []
for path in sorted(p for p in h5 if 'friends' in p):
    with h5py.File(path, 'r') as f:
        key = STIMULUS if STIMULUS else list(f.keys())[0]
        node = f[key]
        data = node[list(node.keys())[0]][:] if isinstance(node, h5py.Group) else node[:]
    # encoder_validation expects (units, timepoints).
    if data.shape[0] > data.shape[1]:
        data = data.T
    responses.append(np.asarray(data, dtype=float))
    print(os.path.basename(path), '->', data.shape)

shortest = min(r.shape[1] for r in responses)
responses = [r[:, :shortest] for r in responses]
print('\nsubjects:', len(responses), 'parcels:', responses[0].shape[0],
      'timepoints:', shortest)

In [ ]:
import json

ceiling = noise_ceiling(responses)
per_subject = np.nanmean(ceiling['per_subject_r'], axis=1)
interval = bootstrap_ci(per_subject, n_resamples=10000, seed=0)

print('Leave-one-subject-out noise ceiling, parcel level')
print(f"  subjects        : {ceiling['n_subjects']}")
print(f"  parcels defined : {ceiling['n_defined']}")
print(f"  mean ceiling    : {ceiling['mean_ceiling']:+.4f}")
print(f"  95% CI over subjects: [{interval['low']:+.4f}, {interval['high']:+.4f}]")
print()
print('Per subject against the mean of the others:')
for i, value in enumerate(per_subject):
    print(f'  subject {i + 1}: {value:+.4f}')

result = {
    'n_subjects': ceiling['n_subjects'],
    'n_parcels_defined': ceiling['n_defined'],
    'mean_ceiling': ceiling['mean_ceiling'],
    'ci_low': interval['low'],
    'ci_high': interval['high'],
    'per_subject': per_subject.tolist(),
    'space': 'Schaefer 1000 parcels, MNI152NLin2009cAsym',
    'note': 'parcel level, not vertex level; averaging within a parcel raises r',
}
with open('/kaggle/working/noise_ceiling.json', 'w') as handle:
    json.dump(result, handle, indent=2)
print('\nwrote /kaggle/working/noise_ceiling.json')

## What this pass establishes, and what it does not

It measures how much of the response is shared between people watching the same
stimulus, in parcel space. That is the number any encoder has to be judged
against, and it is measured here rather than quoted from the audit.

It says nothing yet about the checkpoint. That needs predictions for the same
stimulus, which is a GPU pass over the episode, and it is the second notebook.

A parcel-level correlation is not comparable with the audit's vertex-level
`r = -0.0145`: averaging within a parcel cancels independent noise and raises
correlations. Both numbers are worth having; conflating them would manufacture
an agreement or a disagreement that is really a change of spatial unit.